# Lekcija 03 - Agentni dizajnerski obrasci

U ovoj lekciji istražujemo tri temeljna dizajnerska obrasca za izgradnju učinkovitih AI agenata:

1. **Jasne upute za agenta** — Izrada preciznih, uloge-definirajućih uputa koje usmjeravaju ponašanje agenta  
2. **Strukturirani izlaz s Pydantic modelima** — Osiguravanje da agenti vraćaju predvidljive, validirane podatke  
3. **Agenti s jednom odgovornošću** — Dizajniranje fokusiranih agenata koji svaki rade jednu stvar dobro

Svaki ćemo obrazac primijeniti na scenarij **preporučitelja putnih odredišta**, postupno gradeći sustav koji može predlagati odredišta, provjeravati dostupnost i rješavati logistiku.


## Postavljanje


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## Uzorak 1: Jasne upute za agenta

Najutjecajniji uzorak je i najjednostavniji: pisanje jasnih, detaljnih uputa za vašeg agenta.

Dobre upute definiraju:
- **Tko** je agent (osobnost i ton)
- **Što** treba raditi (odgovornosti korak po korak)
- **Kako** se treba ponašati (ograničenja i stil)

Ispod kreiramo agenta concierge za putovanja s eksplicitnim uputama koje oblikuju svaki odgovor koji generira.


In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## Uzorak 2: Strukturirani izlaz s Pydantic modelima

Tekst slobodnog oblika koristan je za konverzaciju, ali nizvodni sustavi trebaju strukturirane podatke.  
Povezujući **Pydantic modele** s **alatom funkcijom**, možemo:

- Definirati točnu shemu za izlaz agenta  
- Automatski provjeravati valjanost odgovora  
- Pouzdano integrirati rezultate agenta u logiku aplikacije  

Također uvodimo alat koji vraća detalje destinacije kako bi agent utemeljio svoje preporuke na stvarnim podacima.


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(destination: Annotated[str, "The destination to look up"]) -> str:
    """Get details about a vacation destination."""
    details = {
        "Barcelona": "Available. Best: May-Jun. Beach, architecture, nightlife. ~$2000/week",
        "Tokyo": "Available. Best: Mar-Apr. Culture, food, technology. ~$2500/week",
        "Cape Town": "Not available. Best: Nov-Mar. Nature, wine, adventure. ~$1800/week",
    }
    return details.get(destination, f"{destination}: No information available.")


structured_agent = await provider.create_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget"
)

if response:
    print(response)

## Uzorak 3: Agent s jednom odgovornošću

Složeni zadaci imaju koristi od podjele rada na više fokusiranih agenata, svaki sa svojom jedinstvenom odgovornošću:

- **Stručnjak za odredište** koji poznaje mjesta i dostupnost
- **Planer logistike** koji se bavi letovima, hotelima i itinerarima

Ovo odražava načelo softverskog inženjerstva *razdvajanja odgovornosti* — svaki agent je lakše testirati, održavati i poboljšavati neovisno.


In [ ]:
destination_agent = await provider.create_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = await provider.create_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## Sažetak

U ovoj lekciji primijenili smo tri agentička dizajnerska obrasca na scenario preporuke putovanja:

| Obračun | Ključna ideja | Korist |
|---|---|---|
| **Jasne upute** | Definirati personu, odgovornosti i ograničenja unaprijed | Dosljedno, u skladu s brendom ponašanje agenta |
| **Strukturirani izlaz** | Koristiti Pydantic modele kao format odgovora | Validirani, strojno čitljivi rezultati |
| **Jedinstvena odgovornost** | Svakom agentu dati jedan usredotočeni zadatak | Lakše testirati, održavati i kombinirati |

Ovi obrasci se prirodno slažu — možete kombinirati jasne upute sa strukturiranim izlazom unutar agenta s jedinstvenom odgovornošću za izgradnju robusnih, spremnih za proizvodne sustave.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Izjava o odricanju od odgovornosti**:
Ovaj dokument preveden je korištenjem AI prevoditeljske usluge [Co-op Translator](https://github.com/Azure/co-op-translator). Iako nastojimo osigurati točnost, imajte na umu da automatski prijevodi mogu sadržavati pogreške ili netočnosti. Izvorni dokument na izvornom jeziku treba smatrati autoritativnim izvorom. Za kritične informacije preporučuje se profesionalni prijevod od strane stručnog prevoditelja. Nismo odgovorni za bilo kakva nesporazuma ili pogrešne interpretacije koje proizlaze iz korištenja ovog prijevoda.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
